# Analysis of Factors Influencing Airline Ticket Prices

This project investigates the factors associated with average airline fares and evaluates the ability of statistical and machine learning models to predict fares using market-, route-, carrier-, and competition-related characteristics.

The analysis uses the **Airline Market Fare Prediction Data** dataset developed by K. Gülnaz Bülbül from publicly available U.S. Bureau of Transportation Statistics (BTS) data.

**Research objective**

The analysis addresses two related questions:

1. Which market, route, carrier, and competition-related factors are associated with average airline fares?
2. How accurately can statistical and machine learning regression models predict average fares?

The workflow includes exploratory data analysis, feature assessment, regression modeling, ensemble methods, model comparison, feature importance, and robustness analysis.

> This project uses publicly available market-level data. It does not contain confidential or proprietary data from Azerbaijan Airlines or any other airline.

## 1. Setup

This section imports the libraries used for data manipulation and visualization, defines reproducibility settings, and establishes the project paths used throughout the notebook.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
RANDOM_STATE = 42

TARGET = "Average_Fare"
MODELING_SAMPLE_SIZE = 200_000

In [3]:
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.4f}".format)

sns.set_theme(style="whitegrid")

In [4]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "MarketFarePredictionData.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset path: {DATA_PATH}")

Project root: c:\Users\eldar\Desktop\portfolio\python\Analysis of Factors Influencing Airline Ticket Prices Using Statistical and Machine Learning Methods
Dataset path: c:\Users\eldar\Desktop\portfolio\python\Analysis of Factors Influencing Airline Ticket Prices Using Statistical and Machine Learning Methods\data\MarketFarePredictionData.csv


## 2. Data Loading and Validation

The dataset contains airline market observations constructed from the DB1B and T-100 datasets provided by the U.S. Bureau of Transportation Statistics. The processed dataset contains 26 variables covering route characteristics, passenger volumes, carrier activity, market concentration, competition, and average fares.

Before analysis, the dataset structure is validated to ensure that the expected variables are available.

In [5]:
EXPECTED_COLUMNS = [
    "MktCoupons",
    "OriginCityMarketID",
    "DestCityMarketID",
    "OriginAirportID",
    "DestAirportID",
    "Carrier",
    "NonStopMiles",
    "RoundTrip",
    "ODPairID",
    "Pax",
    "CarrierPax",
    "Average_Fare",
    "Market_share",
    "Market_HHI",
    "LCC_Comp",
    "Multi_Airport",
    "Circuity",
    "Slot",
    "Non_Stop",
    "MktMilesFlown",
    "OriginCityMarketID_freq",
    "DestCityMarketID_freq",
    "OriginAirportID_freq",
    "DestAirportID_freq",
    "Carrier_freq",
    "ODPairID_freq",
]

In [6]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. "
        "Download it using the instructions in data/README.md."
    )

df = pd.read_csv(DATA_PATH)

missing_columns = sorted(set(EXPECTED_COLUMNS) - set(df.columns))
unexpected_columns = sorted(set(df.columns) - set(EXPECTED_COLUMNS))

if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

if unexpected_columns:
    print(f"Unexpected columns detected: {unexpected_columns}")

Dataset loaded: 1,581,278 rows × 26 columns


In [7]:
dataset_overview = pd.Series(
    {
        "Rows": f"{len(df):,}",
        "Columns": df.shape[1],
        "Memory usage (MB)": f"{df.memory_usage(deep=True).sum() / 1024**2:,.1f}",
        "Target": TARGET,
    },
    name="Value",
)

display(dataset_overview.to_frame())
display(df.head())

,Value
Rows,"1,581,278"
Columns,26
Memory usage (MB),313.7
Target,Average_Fare


,MktCoupons,OriginCityMarketID,DestCityMarketID,OriginAirportID,DestAirportID,Carrier,NonStopMiles,RoundTrip,ODPairID,Pax,CarrierPax,Average_Fare,Market_share,Market_HHI,LCC_Comp,Multi_Airport,Circuity,Slot,Non_Stop,MktMilesFlown,OriginCityMarketID_freq,DestCityMarketID_freq,OriginAirportID_freq,DestAirportID_freq,Carrier_freq,ODPairID_freq
0,2,178,152,170,255,6,"1,807.0000",1.0000,4035,136.0000,96.0000,389.1000,0.7059,"5,847.7500",1,1,1.3675,0,0.0000,"1,992.4498",0.0041,0.0398,0.0041,0.0220,0.1168,0.0001
1,2,178,152,170,194,20,"1,798.0000",1.0000,4035,136.0000,40.0000,283.3200,0.2941,"5,847.7500",1,1,1.0517,0,0.0000,"1,992.4498",0.0041,0.0398,0.0041,0.0084,0.3077,0.0001
2,2,178,152,170,260,6,"1,784.0000",0.0000,4035,136.0000,96.0000,389.1000,0.7059,"5,847.7500",1,1,1.0348,0,0.0000,"1,992.4498",0.0041,0.0398,0.0041,0.0094,0.1168,0.0001
3,2,178,152,170,255,6,"1,807.0000",1.0000,4035,136.0000,96.0000,389.1000,0.7059,"5,847.7500",1,1,1.0299,0,0.0000,"1,992.4498",0.0041,0.0398,0.0041,0.0220,0.1168,0.0001
4,2,178,152,170,194,20,"1,798.0000",1.0000,4035,136.0000,40.0000,283.3200,0.2941,"5,847.7500",1,1,1.0623,0,0.0000,"1,992.4498",0.0041,0.0398,0.0041,0.0084,0.3077,0.0001
